In [0]:
# =============================================================================
# NB16 — Band gap with external experimental data (transfer / augmentation)
# =============================================================================
# Context (NB15/NB15b): the measured band gap is reproducible across papers
# (median inter-paper SD 0.14 eV → measurement ceiling R² ≈ 0.9), yet the
# model trained on the project's 413 samples reaches only R² ≈ 0.14 on unseen
# papers with composition purging. Hypothesis: the limit is CHEMICAL COVERAGE.
# Test: add a large external experimental band-gap set and check whether
# predictions for the project's perovskites improve under the same strict CV.
#
# External data (verified 2026-10-14):
#   matbench_expt_gap (Matbench v0.1), 4,604 compositions, experimental gap
#   in eV, one value per composition (conflicting duplicates removed by the
#   Matbench curators). Derived from: Y. Zhuo, A. Mansouri Tehrani,
#   J. Brgoch, J. Phys. Chem. Lett. 2018, 9(7), 1668–1673,
#   DOI 10.1021/acs.jpclett.8b00124. Benchmark: Dunn et al., npj Comput.
#   Mater. 2020, 6, 138, DOI 10.1038/s41524-020-00406-3.
#   URL and SHA-256 taken from matminer's dataset_metadata.json.
#   PROVENANCE FLAG: compilation without per-entry references → confidence
#   "medium"; used ONLY for pre-training / as a feature, never merged into the
#   project dataset as curated rows.
#
# Leakage controls
#   - Every external entry whose reduced composition matches ANY composition
#     of the project dataset is removed before training (zero-shot is truly
#     zero-shot; no project label can enter through the external set).
#   - Project evaluation: DOI-grouped 5×5 CV WITH composition purging (as
#     NB15b) — no paper and no composition shared between train and validation.
#   - Same featurisation for both datasets (fraction-weighted statistics of
#     atom_props_curated), built from formula strings in this notebook.
#
# Configurations evaluated on the project samples
#   mean                — training-fold mean (reference)
#   LOCAL               — XGBoost / RF trained on project training folds only
#   EXT_ZERO_SHOT       — XGBoost trained on external data only
#   EXT_CALIBRATED      — zero-shot prediction linearly recalibrated on the
#                         project training fold (2 parameters)
#   STACK               — LOCAL features + zero-shot prediction as a feature
#   COMBINED            — XGBoost trained on external ∪ project training fold
#
# Inputs:  ml_perovskites.master, ml_perovskites.atom_props_curated,
#          external file (downloaded or uploaded, see section 1)
# Outputs: ml_perovskites.ext_bandgap_matbench (external data + provenance),
#          ml_perovskites.bandgap_transfer_nb16, /tmp/nb16_transfer.png
# SEED = 42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import gzip, hashlib, io, json, os, re, subprocess, sys, urllib.request
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute       import SimpleImputer
from sklearn.ensemble     import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics      import r2_score, mean_absolute_error
from sklearn.model_selection import KFold
from pyspark.sql import SparkSession

SEED, SCHEMA, N_FOLDS, N_REPEATS, N_BOOT = 42, "ml_perovskites", 5, 5, 1000

# Two verified sources (URL + SHA-256 from matminer's dataset_metadata.json).
# matbench_expt_gap is preferred (already deduplicated by Matbench curators);
# expt_gap (figshare, 6,354 rows with conflicting duplicates) is the fallback
# and is deduplicated below.
SOURCES = [
    {"name": "matbench_expt_gap",
     "url": "https://ml.materialsproject.org/projects/matbench_expt_gap.json.gz",
     "sha256": "783e7d1461eb83b00b2f2942da4b95fda5e58a0d1ae26b581c24cf8a82ca75b2",
     "needs_dedup": False},
    {"name": "expt_gap",
     "url": "https://ndownloader.figshare.com/files/13464434",
     "sha256": "2d0980e3533c1ba6ad6e392a88f08cfcf2d311d4b7fe6eb0b0c8e876211dfda3",
     "needs_dedup": True},
]
# Manual upload: set this to the uploaded file path (DBFS or Unity Catalog
# Volume), e.g. "/Volumes/workspace/default/raw/matbench_expt_gap.json.gz".
EXT_LOCAL_PATH = ""
LOCAL_CANDIDATES = ["/dbfs/FileStore/matbench_expt_gap.json.gz",
                    "/dbfs/FileStore/tables/matbench_expt_gap.json.gz",
                    "/tmp/matbench_expt_gap.json.gz"]
DEDUP_MAX_SPAN_EV = 0.1   # fallback only: drop compositions whose values span more

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

# ---------------------------------------------------------------------------
# 1. Get the external file (local copy first, then download) and verify hash
# ---------------------------------------------------------------------------
import glob
HASHES = {s["sha256"]: s for s in SOURCES}

def identify(blob):
    """Return the verified source dict matching this file's SHA-256, else None."""
    return HASHES.get(hashlib.sha256(blob).hexdigest())

raw, src, source = None, None, None
# 1) Local copies (explicit path, known paths, any Volume)
candidates = ([EXT_LOCAL_PATH] if EXT_LOCAL_PATH else []) + LOCAL_CANDIDATES + \
             glob.glob("/Volumes/*/*/*/matbench_expt_gap.json.gz") + \
             glob.glob("/Volumes/*/*/*/13464434*") + \
             glob.glob("/Volumes/*/*/*/expt_gap*.json.gz")
for p in candidates:
    if p and os.path.exists(p):
        blob = open(p, "rb").read()
        s = identify(blob)
        if s:
            raw, src, source = blob, p, s
            break
        print(f"  {p}: SHA-256 does not match any verified release — ignored")

# 2) Download (browser-like User-Agent; some servers reject python-urllib)
if raw is None:
    errors = []
    for s in SOURCES:
        try:
            req = urllib.request.Request(s["url"], headers={
                "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
                "Accept": "*/*"})
            with urllib.request.urlopen(req, timeout=120) as r:
                blob = r.read()
            if identify(blob) is not s:
                errors.append(f"{s['name']}: downloaded but SHA-256 mismatch")
                continue
            raw, src, source = blob, s["url"], s
            try:
                os.makedirs("/dbfs/FileStore", exist_ok=True)
                open(f"/dbfs/FileStore/{s['name']}.json.gz", "wb").write(blob)
            except Exception:
                pass
            break
        except Exception as e:
            errors.append(f"{s['name']}: {e}")
    if raw is None:
        raise RuntimeError(
            "No verified external file available.\n  Download attempts: " +
            " | ".join(errors) +
            "\n  Manual route: download one of\n    " +
            "\n    ".join(s["url"] for s in SOURCES) +
            "\n  in a browser, upload it to a Volume (Catalog → your volume → Upload) or "
            "DBFS, and set EXT_LOCAL_PATH at the top of this notebook.")

print(f"External file OK — {source['name']} (SHA-256 verified) from {src}")

obj = json.loads(gzip.decompress(raw).decode("utf-8"))
if isinstance(obj, dict) and "data" in obj and "columns" in obj:
    ext = pd.DataFrame(obj["data"], columns=obj["columns"])
elif isinstance(obj, list):
    ext = pd.DataFrame(obj)
else:
    ext = pd.DataFrame(obj)
comp_col = "composition" if "composition" in ext.columns else "formula"
gap_col = "gap expt"

def comp_to_str(v):
    """Matbench may store compositions as strings or serialized dicts."""
    if isinstance(v, dict):
        items = {k: v[k] for k in v if not str(k).startswith("@")}
        return "".join(f"{k}{float(a):g}" for k, a in items.items())
    return str(v)

ext = pd.DataFrame({"formula": ext[comp_col].map(comp_to_str),
                    "eg": pd.to_numeric(ext[gap_col], errors="coerce")}).dropna()
print(f"External entries: {len(ext)} | Eg {ext['eg'].min():.2f}–{ext['eg'].max():.2f} eV, "
      f"zero-gap entries: {int((ext['eg'] == 0).sum())}")
# Deduplication of the fallback source happens after parsing (section 3),
# on reduced compositions, so "SrTiO3" and "Sr1Ti1O3" are treated as one.

# ---------------------------------------------------------------------------
# 2. Formula parser + featuriser (shared by both datasets)
# ---------------------------------------------------------------------------
ap = spark.table(f"{SCHEMA}.atom_props_curated").toPandas().set_index("Element")
for c in ap.columns:
    ap[c] = pd.to_numeric(ap[c], errors="coerce").astype(float)
PROPS = [c for c in ap.columns if ap[c].notna().mean() > 0.5]
ELEMENTS = set(ap.index)

TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)|(\()|(\))(\d*\.?\d*)")

def clean_formula(s):
    s = str(s)
    s = re.sub(r"\([^)]*\s[^)]*\)", "", s)          # annotations with spaces: "(1073 K)"
    s = re.sub(r"\([^)]*[a-z]{2,}[^)]*\)", "", s)   # word annotations: "(partial)"
    return re.sub(r"\s+", "", s)

def parse(s):
    s = clean_formula(s)
    if not s or "/" in s or "-" in s:
        return None                      # composites / ambiguous strings: skip
    stack, i = [{}], 0
    while i < len(s):
        m = TOKEN.match(s, i)
        if not m:
            return None
        if m.group(1):
            el, n = m.group(1), m.group(2)
            if el not in ELEMENTS:
                return None
            stack[-1][el] = stack[-1].get(el, 0.0) + (float(n) if n not in ("", ".") else 1.0)
        elif m.group(3):
            stack.append({})
        else:
            if len(stack) < 2:
                return None
            mult = float(m.group(5)) if m.group(5) not in ("", ".") else 1.0
            top = stack.pop()
            for k, v in top.items():
                stack[-1][k] = stack[-1].get(k, 0.0) + v * mult
        i = m.end()
    if len(stack) != 1 or not stack[0] or sum(stack[0].values()) <= 0:
        return None
    tot = sum(stack[0].values())
    return {k: v / tot for k, v in stack[0].items()}

def comp_key(fr):
    return "".join(f"{k}{v:.3f}" for k, v in sorted(fr.items()))

def featurise(fr):
    out = {"n_elements": len(fr)}
    els, x = list(fr), np.array(list(fr.values()))
    for p in PROPS:
        v = ap.loc[els, p].values
        ok = ~np.isnan(v)
        if not ok.any():
            for s in ["mean", "std", "min", "max", "range"]:
                out[f"{p}_{s}"] = np.nan
            continue
        w = x[ok] / x[ok].sum()
        mu = float(np.sum(w * v[ok]))
        out[f"{p}_mean"] = mu
        out[f"{p}_std"] = float(np.sqrt(np.sum(w * (v[ok] - mu) ** 2)))
        out[f"{p}_min"] = float(v[ok].min())
        out[f"{p}_max"] = float(v[ok].max())
        out[f"{p}_range"] = float(v[ok].max() - v[ok].min())
    return out

def build(df, formula_col):
    fr = df[formula_col].map(parse)
    ok = fr.notna()
    feats = pd.DataFrame([featurise(f) for f in fr[ok]], index=df.index[ok])
    keys = fr[ok].map(comp_key)
    return ok, feats, keys

# ---------------------------------------------------------------------------
# 3. Project samples (same dedup as NB15/NB15b), featurised by base_compound
# ---------------------------------------------------------------------------
master = spark.table(f"{SCHEMA}.master").toPandas()
master["eg"] = pd.to_numeric(master["bandgap_eV"], errors="coerce")
d = master[master["eg"].between(0.0, 6.5) & master["doi_key"].notna() &
           master["base_compound"].notna()].copy()
sample_key = "Compound_original" if "Compound_original" in d.columns else "base_compound"
d = (d.groupby(["doi_key", sample_key], as_index=False)
       .agg(eg=("eg", "mean"), base_compound=("base_compound", "first")))
ok, Fp, Kp = build(d, "base_compound")
print(f"\nProject samples: {len(d)} | parsed: {int(ok.sum())} | unparsed (composites / "
      f"text): {int((~ok).sum())} e.g. {d.loc[~ok, 'base_compound'].unique()[:6].tolist()}")
d = d[ok].copy()
d["key"] = Kp

okx, Fx, Kx = build(ext, "formula")
ext = ext[okx].copy()
ext["key"] = Kx
print(f"External parsed: {len(ext)} / {int(okx.size)}")

if source["needs_dedup"]:
    # Fallback source: conflicting duplicates. Approximates the Matbench rule as
    # described by its maintainers (not an exact reproduction): drop compositions
    # whose reported values span > DEDUP_MAX_SPAN_EV, keep the median otherwise.
    g = ext.groupby("key")["eg"]
    span = g.transform(lambda s: s.max() - s.min())
    n_before, n_keys = len(ext), ext["key"].nunique()
    conflicted = ext.loc[span > DEDUP_MAX_SPAN_EV, "key"].nunique()
    ext = ext[span <= DEDUP_MAX_SPAN_EV].copy()
    ext["eg"] = ext.groupby("key")["eg"].transform("median")
    ext = ext.drop_duplicates("key")
    print(f"Dedup ({source['name']}): {n_before} rows / {n_keys} compositions → "
          f"{len(ext)} compositions ({conflicted} dropped as conflicting, span > "
          f"{DEDUP_MAX_SPAN_EV} eV)")
Fx = Fx.loc[ext.index]

FEATS = sorted(set(Fp.columns) | set(Fx.columns))
Fp, Fx = Fp.reindex(columns=FEATS), Fx.reindex(columns=FEATS)

# ---------------------------------------------------------------------------
# 4. Cross-database agreement on shared compositions, then purge them
# ---------------------------------------------------------------------------
proj_keys = set(d["key"])
shared = ext[ext["key"].isin(proj_keys)]
if len(shared):
    m = (d.groupby("key")["eg"].mean().rename("eg_project").to_frame()
           .join(shared.groupby("key")["eg"].mean().rename("eg_external"), how="inner"))
    print(f"\nShared compositions: {len(m)} | |project − external| median "
          f"{(m['eg_project'] - m['eg_external']).abs().median():.3f} eV, "
          f"mean bias (project − external) {(m['eg_project'] - m['eg_external']).mean():+.3f} eV")
    print(m.round(2).head(15).to_string())
ext_p = ext[~ext["key"].isin(proj_keys)].copy()
Fx_p = Fx.loc[ext_p.index]
print(f"External after purging project compositions: {len(ext_p)} (removed {len(shared)})")

# Persist external data with provenance (separate table, never merged)
ext_out = ext[["formula", "eg", "key"]].copy()
ext_out["source"] = (f"{source['name']} via matminer (Zhuo et al. 2018, "
                     f"doi:10.1021/acs.jpclett.8b00124)"
                     + ("; deduplicated in NB16" if source["needs_dedup"] else ""))
ext_out["confidence"] = "medium (compilation without per-entry references)"
ext_out["shared_with_project"] = ext_out["key"].isin(proj_keys)
spark.createDataFrame(ext_out).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.ext_bandgap_matbench")

# ---------------------------------------------------------------------------
# 5. Sanity check of the featuriser on the external set (random 5-fold CV)
# ---------------------------------------------------------------------------
def xgb_model(n=600):
    return xgb.XGBRegressor(n_estimators=n, learning_rate=0.05, max_depth=6,
                            subsample=0.8, colsample_bytree=0.6, min_child_weight=2,
                            tree_method="hist", random_state=SEED, verbosity=0)

Xe, ye = Fx_p.values.astype(float), ext_p["eg"].values
imp_e = SimpleImputer(strategy="median").fit(Xe)
Xe_i = imp_e.transform(Xe)
pe = np.zeros(len(ye))
for tr, va in KFold(5, shuffle=True, random_state=SEED).split(Xe_i):
    pe[va] = xgb_model().fit(Xe_i[tr], ye[tr]).predict(Xe_i[va])
print(f"\nExternal-set sanity (random 5-fold, XGBoost): MAE {mean_absolute_error(ye, pe):.3f} eV, "
      f"R² {r2_score(ye, pe):.3f}  (Matbench leaderboard reports MAE ≈ 0.29–0.33 eV for the "
      f"best composition-only models under its own protocol — order-of-magnitude check only)")

# Zero-shot model: trained once on ALL purged external data
ext_model = xgb_model().fit(Xe_i, ye)

# ---------------------------------------------------------------------------
# 6. Project evaluation — DOI-grouped CV with composition purging
# ---------------------------------------------------------------------------
Xp = Fp.loc[d.index].values.astype(float)
yp = d["eg"].values
zs = ext_model.predict(imp_e.transform(Xp))          # zero-shot prediction (no leakage)
dois = np.array(sorted(d["doi_key"].unique()))
comps = d["base_compound"].values

rows, oof0 = [], {}
CONFIGS = ["mean", "LOCAL_XGB", "LOCAL_RF", "EXT_ZERO_SHOT", "EXT_CALIBRATED",
           "STACK_XGB", "COMBINED_XGB"]
for rep in range(N_REPEATS):
    perm = np.random.default_rng(SEED + rep).permutation(dois)
    f_of = {g: i % N_FOLDS for i, g in enumerate(perm)}
    f = d["doi_key"].map(f_of).values
    P = {c: np.full(len(d), np.nan) for c in CONFIGS}
    for k in range(N_FOLDS):
        va = np.where(f == k)[0]
        tr = np.where(f != k)[0]
        tr = tr[~np.isin(comps[tr], list(set(comps[va])))]      # composition purge
        if len(tr) < 30:
            print(f"  WARNING repeat {rep} fold {k}: only {len(tr)} training samples after "
                  f"purge — fold skipped")
            continue
        imp = SimpleImputer(strategy="median").fit(Xp[tr])
        Xtr, Xva = imp.transform(Xp[tr]), imp.transform(Xp[va])
        P["mean"][va] = yp[tr].mean()
        P["LOCAL_XGB"][va] = xgb_model(400).fit(Xtr, yp[tr]).predict(Xva)
        P["LOCAL_RF"][va] = RandomForestRegressor(n_estimators=500, min_samples_leaf=2,
                                                  max_features=0.3, n_jobs=-1,
                                                  random_state=SEED).fit(Xtr, yp[tr]).predict(Xva)
        P["EXT_ZERO_SHOT"][va] = zs[va]
        cal = LinearRegression().fit(zs[tr].reshape(-1, 1), yp[tr])
        P["EXT_CALIBRATED"][va] = cal.predict(zs[va].reshape(-1, 1))
        P["STACK_XGB"][va] = xgb_model(400).fit(np.c_[Xtr, zs[tr]], yp[tr]) \
                                           .predict(np.c_[Xva, zs[va]])
        Xc = np.r_[Xe_i, imp_e.transform(Xp[tr])]
        yc = np.r_[ye, yp[tr]]
        P["COMBINED_XGB"][va] = xgb_model().fit(Xc, yc).predict(imp_e.transform(Xp[va]))
    for c, p in P.items():
        okp = ~np.isnan(p)
        rows.append({"repeat": rep, "config": c, "r2": r2_score(yp[okp], p[okp]),
                     "mae_eV": mean_absolute_error(yp[okp], p[okp]), "n_pred": int(okp.sum())})
        if rep == 0:
            oof0[c] = p
    print(f"  repeat {rep + 1}/{N_REPEATS} done")

# restrict bootstrap/plots to samples predicted in repeat 0 by every config
valid0 = np.all([~np.isnan(p) for p in oof0.values()], axis=0)

res = pd.DataFrame(rows)
summ = res.groupby("config").agg(r2=("r2", "mean"), r2_min=("r2", "min"),
                                 r2_max=("r2", "max"), mae_eV=("mae_eV", "mean"))
by = {g: np.where((d["doi_key"].values == g) & valid0)[0] for g in dois}
by = {g: v for g, v in by.items() if len(v)}
dois_b = np.array(list(by))
rng = np.random.default_rng(SEED + 3)
boot_idx = [np.concatenate([by[g] for g in rng.choice(dois_b, len(dois_b), replace=True)])
            for _ in range(N_BOOT)]
ci = {c: (np.percentile([r2_score(yp[i], p[i]) for i in boot_idx], 2.5),
          np.percentile([r2_score(yp[i], p[i]) for i in boot_idx], 97.5)) for c, p in oof0.items()}
summ["ci_lo"] = [ci[c][0] for c in summ.index]
summ["ci_hi"] = [ci[c][1] for c in summ.index]
# paired gain vs LOCAL_XGB on the same bootstrap resamples
for c in CONFIGS:
    dlt = [r2_score(yp[i], oof0[c][i]) - r2_score(yp[i], oof0["LOCAL_XGB"][i]) for i in boot_idx]
    summ.loc[c, "dR2_vs_local"] = np.mean(dlt)
    summ.loc[c, "dR2_p2.5"] = np.percentile(dlt, 2.5)
    summ.loc[c, "dR2_p97.5"] = np.percentile(dlt, 97.5)
summ = summ.reindex(CONFIGS)
pd.set_option("display.width", 220)
print(f"\n--- Project band gap, DOI-grouped CV with composition purging ({N_REPEATS}×{N_FOLDS}); "
      f"CI and ΔR² = cluster bootstrap by DOI (repeat 0) ---")
print(summ.round(3).to_string())
print("\nReading guide:")
print("  * dR2 CI above 0 for STACK/COMBINED/EXT_* → external coverage helps (coverage hypothesis supported).")
print("  * EXT_CALIBRATED >> EXT_ZERO_SHOT → systematic offset between photocatalysis-paper Eg and the compilation.")

# ---------------------------------------------------------------------------
# 7. Plot + write
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
lim = (max(0, yp.min() - 0.2), yp.max() + 0.2)
for ax, c in zip(axes, ["LOCAL_XGB", "EXT_ZERO_SHOT", "COMBINED_XGB"]):
    ax.scatter(yp[valid0], oof0[c][valid0], s=12, alpha=0.6)
    ax.plot(lim, lim, color="black", lw=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Measured Eg (eV)"); ax.set_ylabel("Predicted Eg (eV)")
    s = summ.loc[c]
    ax.set_title(f"{c}: R² {s['r2']:.2f} [{s['ci_lo']:.2f}, {s['ci_hi']:.2f}]", fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/nb16_transfer.png", dpi=130); plt.close()
print("\nSaved: /tmp/nb16_transfer.png")

spark.createDataFrame(res).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.bandgap_transfer_nb16")
print(f"✓ Written: {SCHEMA}.ext_bandgap_matbench ({len(ext_out)} rows), "
      f"{SCHEMA}.bandgap_transfer_nb16 ({len(res)} rows)")
print("\n=== NB16 COMPLETE ===")